# Set up environment

⚠️ Must restart session after installing HMMER package in order for ANARCI to install and run correctly.

⏳ 5 minutes

In [1]:
# @title Mount Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# @title Install Packages
!pip install biopython #required by ANARCI
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install tensorflow
!pip install matplotlib
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.8 MB/s eta 0:00:00


In [3]:
# @title Install HMMER
!apt-get install -y hmmer #required by ANARCI

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libdivsufsort3
Suggested packages:
  hmmer-doc
The following NEW packages will be installed:
  hmmer libdivsufsort3
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,198 kB of archives.
After this operation, 7,621 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libdivsufsort3 amd64 2.0.1-5 [42.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 hmmer amd64 3.3.2+dfsg-1 [1,155 kB]
Fetched 1,198 kB in 1s (825 kB/s)
Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_amd64

In [ ]:
# @title Restart Session
import os
os.kill(os.getpid(), 9)

In [1]:
# @title  Clone and install ANARCI
!git clone https://github.com/oxpig/ANARCI.git
%cd ANARCI
!python setup.py install

Cloning into 'ANARCI'...
remote: Enumerating objects: 793, done.
remote: Counting objects: 100% (293/293), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 793 (delta 249), reused 210 (delta 210), pack-reused 500 (from 2)
Receiving objects: 100% (793/793), 6.52 MiB | 20.11 MiB/s, done.
Resolving deltas: 100% (454/454), done.
/content/ANARCI
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
INFO: ANARCI lives in:  /usr/local/lib/python3.12/dist-p

In [2]:
# @title Load Libraries

# ===== Data tools =====
import pandas as pd
import numpy as np
import csv
import os
import gc
import joblib

# ===== Bioinformatics =====
from anarci import anarci
from collections import Counter

# ===== Visualization =====
import matplotlib.pyplot as plt
import seaborn as sns

# ===== Deep learning tools =====
import torch
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# ===== Preprocessing & Evaluation =====
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

# ===== Models =====
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

# Sequence Alignment

Goal: Numbering of protein sequences

⏳ 1 minute

In [3]:
# @title Create a working directory on Drive
output_dir = "/content/drive/MyDrive/poster_antibodies"
!mkdir -p {output_dir}
print(f"Results will be saved to: {output_dir}")

Results will be saved to: /content/drive/MyDrive/poster_antibodies


In [4]:
# @title Clean dataset

# Load CSV file
file_path = "/content/drive/MyDrive/Downloads/Colab/antibody_data.csv"
df = pd.read_csv(file_path)

# Keep only the desired columns
final_df = df[['antibody_id', 'vh_protein_sequence', 'vl_protein_sequence', 'tm2_nanodsf_avg']]

# Total number of antibodies (rows)
total_antibodies = final_df.shape[0]

# Clean dataset
df_clean = final_df.dropna(subset=["tm2_nanodsf_avg"])
df_clean = df_clean.rename(columns={"tm2_nanodsf_avg": "temp"})
remaining_antibodies = df_clean.shape[0]

print(f"Total antibodies: {total_antibodies}")
print(f"After dropping empty Tm values: {remaining_antibodies}")
print(f"Dropped rows: {total_antibodies - remaining_antibodies}")

# Define the output directory
output_dir = "/content/drive/MyDrive/poster_antibodies"

# Save dataset to the specified directory
output_filepath = os.path.join(output_dir, "antibody_temp.csv")
df_clean.to_csv(output_filepath, index=False)
print(f"CSV file '{output_filepath}' created successfully.")
print(f"Number of rows: {len(df_clean)}")

Total antibodies: 246
After dropping empty Tm values: 208
Dropped rows: 38
CSV file '/content/drive/MyDrive/poster_antibodies/antibody_temp.csv' created successfully.
Number of rows: 208


# Physicochemical matrix

Goal: Extract physicochemical and structural features of protein sequences.

⏳ 1 minute

In [5]:
# @title Create a working directory on Drive
output_dir = "/content/drive/MyDrive/physicochemical_antibody_results"
!mkdir -p {output_dir}
print(f"Results will be saved to: {output_dir}")

Results will be saved to: /content/drive/MyDrive/physicochemical_antibody_results


In [6]:
# @title Biophysical Analysis VH sequences

# ===== Function to analyze residues in protein sequence =====

def analyze_amino_acids(sequence):
    # Define the amino acids
    amino_acids = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

    # Convert to uppercase and filter only valid amino acids
    sequence_clean = ''.join([aa for aa in sequence.upper() if aa in amino_acids])

    # Count amino acid occurrences
    aa_count = Counter(sequence_clean)
    total_residues = len(sequence_clean)

    # Calculate raw counts with distinct keys
    aa_counts = {f"{aa}_count": aa_count.get(aa, 0) for aa in amino_acids}

    # Calculate amino acid ratios with distinct keys
    aa_ratios = {f"{aa}_ratio": aa_count.get(aa, 0) / total_residues if total_residues > 0 else 0
                 for aa in amino_acids}

    # Define amino acid categories
    basic = ['K', 'R', 'H']
    acidic = ['D', 'E']
    polar_uncharged = ['S', 'T', 'N', 'Q', 'Y', 'C']
    hydrophobic = ['A', 'V', 'I', 'L', 'M', 'F', 'W', 'P']
    special = ['G']  # Glycine is flexible

    # Create category lists
    charged = basic + acidic
    non_charged = polar_uncharged + hydrophobic + special

    # Calculate category counts
    charged_count = sum(aa_count.get(aa, 0) for aa in charged)
    non_charged_count = sum(aa_count.get(aa, 0) for aa in non_charged)
    basic_count = sum(aa_count.get(aa, 0) for aa in basic)
    acidic_count = sum(aa_count.get(aa, 0) for aa in acidic)
    polar_count = sum(aa_count.get(aa, 0) for aa in polar_uncharged)
    hydrophobic_count = sum(aa_count.get(aa, 0) for aa in hydrophobic)

    # Calculate various ratios
    results = {
        **aa_counts,  # A_count, R_count, etc.
        **aa_ratios,   # A_ratio, R_ratio, etc.

        # Basic statistics
        "sequence_length": len(sequence),
        "valid_amino_acids": total_residues,

        # Category counts
        "charged_count": charged_count,
        "non_charged_count": non_charged_count,
        "basic_count": basic_count,
        "acidic_count": acidic_count,
        "polar_count": polar_count,
        "hydrophobic_count": hydrophobic_count,

        # Category ratios (to total residues)
        "charged_ratio": charged_count / total_residues if total_residues > 0 else 0,
        "non_charged_ratio": non_charged_count / total_residues if total_residues > 0 else 0,
        "basic_ratio": basic_count / total_residues if total_residues > 0 else 0,
        "acidic_ratio": acidic_count / total_residues if total_residues > 0 else 0,
        "polar_ratio": polar_count / total_residues if total_residues > 0 else 0,
        "hydrophobic_ratio": hydrophobic_count / total_residues if total_residues > 0 else 0,

        # Special ratios
        "charged_to_non_charged": charged_count / non_charged_count if non_charged_count > 0 else 0,
        "basic_to_acidic": basic_count / acidic_count if acidic_count > 0 else 0,
        "polar_to_hydrophobic": polar_count / hydrophobic_count if hydrophobic_count > 0 else 0,

        # Specific amino acid group ratios
        "E_F_M_R_ratio": (aa_count.get('E', 0) + aa_count.get('F', 0) +
                          aa_count.get('M', 0) + aa_count.get('R', 0)) / total_residues if total_residues > 0 else 0,

        "aromatic_ratio": (aa_count.get('F', 0) + aa_count.get('W', 0) + aa_count.get('Y', 0)) / total_residues if total_residues > 0 else 0,

        "aliphatic_ratio": (aa_count.get('A', 0) + aa_count.get('V', 0) +
                           aa_count.get('L', 0) + aa_count.get('I', 0)) / total_residues if total_residues > 0 else 0,

        "tiny_ratio": (aa_count.get('A', 0) + aa_count.get('G', 0) + aa_count.get('S', 0)) / total_residues if total_residues > 0 else 0,

        "small_ratio": (aa_count.get('A', 0) + aa_count.get('G', 0) + aa_count.get('S', 0) +
                       aa_count.get('T', 0) + aa_count.get('P', 0)) / total_residues if total_residues > 0 else 0,
    }

    return results


# ===== Function to analyze each protein sequence =====

def analyze_protein_sequences(csv_file, sequence_column):
    # Read CSV
    df = pd.read_csv(csv_file)

    # Create empty list for results
    results_list = []

    # Analyze each sequence
    for idx, row in df.iterrows():
        sequence = row[sequence_column]

        if pd.notna(sequence) and isinstance(sequence, str):
            analysis_results = analyze_amino_acids(sequence)
            results_list.append(analysis_results)
        else:
            # Create empty dict with NaN values
            results_list.append({key: None for key in analyze_amino_acids("")})

    # Create analysis DataFrame
    analysis_df = pd.DataFrame(results_list)

    # Set suffix for DataFrame
    analysis_df = analysis_df.add_suffix("_vh")

    # Concatenate with original
    result_df = pd.concat([df, analysis_df], axis=1)

    # Save results
    result_df.to_csv(os.path.join(output_dir, "analyzed_sequences_vh.csv"), index=False)

    return result_df

# ===== Analysis Results =====

# Run analysis
csv_file = "/content/drive/MyDrive/poster_antibodies/antibody_temp.csv"
sequence_column = "vh_protein_sequence"
output_dir = "/content/drive/MyDrive/physicochemical_antibody_results"

analyzed_df = analyze_protein_sequences(csv_file, sequence_column)

# Display summary
print("=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"Total sequences analyzed: {len(analyzed_df)}")
print(f"Columns in output: {len(analyzed_df.columns)}")
print("\nFirst few rows with key metrics:")
print("=" * 80)

# Show important columns
key_columns = ['sequence_length', 'valid_amino_acids',
               'charged_ratio', 'hydrophobic_ratio',
               'E_F_M_R_ratio', 'aromatic_ratio']

if all(col in analyzed_df.columns for col in key_columns):
    print(analyzed_df[key_columns].head())
else:
    print("Showing available columns:")
    for col in analyzed_df.columns[-20:]:  # Show last 20 columns
        print(f"  {col}")

# Show amino acid counts for first sequence
print("\n" + "=" * 80)
print("AMINO ACID COUNTS for first sequence:")
print("=" * 80)

first_seq = analyzed_df.iloc[0][sequence_column] if sequence_column in analyzed_df.columns else ""
if first_seq:
    aa_counts = {col: analyzed_df.iloc[0][col] for col in analyzed_df.columns if col.endswith('_count')}
    for aa, count in sorted(aa_counts.items()):
        if count > 0:
            print(f"{aa}: {count}")

print("\n Composition for VH saved as CSV: analyzed_sequences_vh.csv")


ANALYSIS COMPLETE!
Total sequences analyzed: 208
Columns in output: 66

First few rows with key metrics:
Showing available columns:
  charged_count_vh
  non_charged_count_vh
  basic_count_vh
  acidic_count_vh
  polar_count_vh
  hydrophobic_count_vh
  charged_ratio_vh
  non_charged_ratio_vh
  basic_ratio_vh
  acidic_ratio_vh
  polar_ratio_vh
  hydrophobic_ratio_vh
  charged_to_non_charged_vh
  basic_to_acidic_vh
  polar_to_hydrophobic_vh
  E_F_M_R_ratio_vh
  aromatic_ratio_vh
  aliphatic_ratio_vh
  tiny_ratio_vh
  small_ratio_vh

AMINO ACID COUNTS for first sequence:

 Composition for VH saved as CSV: analyzed_sequences_vh.csv


In [7]:
# @title Biophysical Analysis VL sequences

# ===== Function to analyze residues in protein sequence =====

def analyze_amino_acids(sequence):
    # Define the amino acids
    amino_acids = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

    # Convert to uppercase and filter only valid amino acids
    sequence_clean = ''.join([aa for aa in sequence.upper() if aa in amino_acids])

    # Count amino acid occurrences
    aa_count = Counter(sequence_clean)
    total_residues = len(sequence_clean)

    # Calculate raw counts with distinct keys
    aa_counts = {f"{aa}_count": aa_count.get(aa, 0) for aa in amino_acids}

    # Calculate amino acid ratios with distinct keys
    aa_ratios = {f"{aa}_ratio": aa_count.get(aa, 0) / total_residues if total_residues > 0 else 0
                 for aa in amino_acids}

    # Define amino acid categories
    basic = ['K', 'R', 'H']
    acidic = ['D', 'E']
    polar_uncharged = ['S', 'T', 'N', 'Q', 'Y', 'C']
    hydrophobic = ['A', 'V', 'I', 'L', 'M', 'F', 'W', 'P']
    special = ['G']  # Glycine is flexible

    # Create category lists
    charged = basic + acidic
    non_charged = polar_uncharged + hydrophobic + special

    # Calculate category counts
    charged_count = sum(aa_count.get(aa, 0) for aa in charged)
    non_charged_count = sum(aa_count.get(aa, 0) for aa in non_charged)
    basic_count = sum(aa_count.get(aa, 0) for aa in basic)
    acidic_count = sum(aa_count.get(aa, 0) for aa in acidic)
    polar_count = sum(aa_count.get(aa, 0) for aa in polar_uncharged)
    hydrophobic_count = sum(aa_count.get(aa, 0) for aa in hydrophobic)

    # Calculate various ratios
    results = {
        **aa_counts,  # A_count, R_count, etc.
        **aa_ratios,   # A_ratio, R_ratio, etc.

        # Basic statistics
        "sequence_length": len(sequence),
        "valid_amino_acids": total_residues,

        # Category counts
        "charged_count": charged_count,
        "non_charged_count": non_charged_count,
        "basic_count": basic_count,
        "acidic_count": acidic_count,
        "polar_count": polar_count,
        "hydrophobic_count": hydrophobic_count,

        # Category ratios (to total residues)
        "charged_ratio": charged_count / total_residues if total_residues > 0 else 0,
        "non_charged_ratio": non_charged_count / total_residues if total_residues > 0 else 0,
        "basic_ratio": basic_count / total_residues if total_residues > 0 else 0,
        "acidic_ratio": acidic_count / total_residues if total_residues > 0 else 0,
        "polar_ratio": polar_count / total_residues if total_residues > 0 else 0,
        "hydrophobic_ratio": hydrophobic_count / total_residues if total_residues > 0 else 0,

        # Special ratios
        "charged_to_non_charged": charged_count / non_charged_count if non_charged_count > 0 else 0,
        "basic_to_acidic": basic_count / acidic_count if acidic_count > 0 else 0,
        "polar_to_hydrophobic": polar_count / hydrophobic_count if hydrophobic_count > 0 else 0,

        # Specific amino acid group ratios
        "E_F_M_R_ratio": (aa_count.get('E', 0) + aa_count.get('F', 0) +
                          aa_count.get('M', 0) + aa_count.get('R', 0)) / total_residues if total_residues > 0 else 0,

        "aromatic_ratio": (aa_count.get('F', 0) + aa_count.get('W', 0) + aa_count.get('Y', 0)) / total_residues if total_residues > 0 else 0,

        "aliphatic_ratio": (aa_count.get('A', 0) + aa_count.get('V', 0) +
                           aa_count.get('L', 0) + aa_count.get('I', 0)) / total_residues if total_residues > 0 else 0,

        "tiny_ratio": (aa_count.get('A', 0) + aa_count.get('G', 0) + aa_count.get('S', 0)) / total_residues if total_residues > 0 else 0,

        "small_ratio": (aa_count.get('A', 0) + aa_count.get('G', 0) + aa_count.get('S', 0) +
                       aa_count.get('T', 0) + aa_count.get('P', 0)) / total_residues if total_residues > 0 else 0,
    }

    return results


# ===== Function to analyze each protein sequence =====

def analyze_protein_sequences(csv_file, sequence_column):
    # Read CSV
    df = pd.read_csv(csv_file)

    # Create empty list for results
    results_list = []

    # Analyze each sequence
    for idx, row in df.iterrows():
        sequence = row[sequence_column]

        if pd.notna(sequence) and isinstance(sequence, str):
            analysis_results = analyze_amino_acids(sequence)
            results_list.append(analysis_results)
        else:
            # Create empty dict with NaN values
            results_list.append({key: None for key in analyze_amino_acids("")})

    # Create analysis DataFrame
    analysis_df = pd.DataFrame(results_list)

    # Set suffix for DataFrame
    analysis_df = analysis_df.add_suffix("_vl")

    # Concatenate with original
    result_df = pd.concat([df, analysis_df], axis=1)

    # Save results
    result_df.to_csv(os.path.join(output_dir, "analyzed_sequences_vl.csv"), index=False)

    return result_df

# ===== Analysis Results =====

# Run analysis
csv_file = "/content/drive/MyDrive/poster_antibodies/antibody_temp.csv"
sequence_column = "vl_protein_sequence"
output_dir = "/content/drive/MyDrive/physicochemical_antibody_results"

analyzed_df = analyze_protein_sequences(csv_file, sequence_column)

# Display summary
print("=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"Total sequences analyzed: {len(analyzed_df)}")
print(f"Columns in output: {len(analyzed_df.columns)}")
print("\nFirst few rows with key metrics:")
print("=" * 80)

# Show important columns
key_columns = ['sequence_length', 'valid_amino_acids',
               'charged_ratio', 'hydrophobic_ratio',
               'E_F_M_R_ratio', 'aromatic_ratio']

if all(col in analyzed_df.columns for col in key_columns):
    print(analyzed_df[key_columns].head())
else:
    print("Showing available columns:")
    for col in analyzed_df.columns[-20:]:  # Show last 20 columns
        print(f"  {col}")

# Show amino acid counts for first sequence
print("\n" + "=" * 80)
print("AMINO ACID COUNTS for first sequence:")
print("=" * 80)

first_seq = analyzed_df.iloc[0][sequence_column] if sequence_column in analyzed_df.columns else ""
if first_seq:
    aa_counts = {col: analyzed_df.iloc[0][col] for col in analyzed_df.columns if col.endswith('_count')}
    for aa, count in sorted(aa_counts.items()):
        if count > 0:
            print(f"{aa}: {count}")

print("\n Composition for VL saved as CSV: analyzed_sequences_vl.csv")


ANALYSIS COMPLETE!
Total sequences analyzed: 208
Columns in output: 66

First few rows with key metrics:
Showing available columns:
  charged_count_vl
  non_charged_count_vl
  basic_count_vl
  acidic_count_vl
  polar_count_vl
  hydrophobic_count_vl
  charged_ratio_vl
  non_charged_ratio_vl
  basic_ratio_vl
  acidic_ratio_vl
  polar_ratio_vl
  hydrophobic_ratio_vl
  charged_to_non_charged_vl
  basic_to_acidic_vl
  polar_to_hydrophobic_vl
  E_F_M_R_ratio_vl
  aromatic_ratio_vl
  aliphatic_ratio_vl
  tiny_ratio_vl
  small_ratio_vl

AMINO ACID COUNTS for first sequence:

 Composition for VL saved as CSV: analyzed_sequences_vl.csv


In [9]:
# @title Merge physicochemical sequences

# Load CSV file
vh = pd.read_csv("/content/drive/MyDrive/physicochemical_antibody_results/analyzed_sequences_vh.csv")
vl = pd.read_csv("/content/drive/MyDrive/physicochemical_antibody_results/analyzed_sequences_vl.csv")

# Merge files
df = pd.merge(
    vh, vl,
    on=["antibody_id", "vh_protein_sequence", "vl_protein_sequence",
        "temp"],
    how="outer"
)

# Save file
df.to_csv(os.path.join(output_dir, "analyzed_sequences_complete.csv"), index=False)